# Colab End-to-End Single-Task Baseline

This notebook runs the whole pipeline on Colab in one place:

1. install dependencies
2. download dataset from Hugging Face
3. sample a lightweight subset
4. normalize schema
5. train a single-task baseline

Supported tasks:
- `docvqa` -> Donut
- `chartqa` -> Pix2Struct

In [ ]:
!python -V
!nvidia-smi

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio transformers datasets pillow sentencepiece accelerate

In [ ]:
import os
from pathlib import Path

import torch
from datasets import concatenate_datasets, load_dataset
from transformers import (
    DonutProcessor,
    Pix2StructForConditionalGeneration,
    Pix2StructProcessor,
    Trainer,
    TrainingArguments,
    VisionEncoderDecoderModel,
)

WORKDIR = Path('/content/moe_single_task')
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print('Working directory:', WORKDIR)

In [ ]:
# Change only these config values.
TASK = 'docvqa'  # 'docvqa' or 'chartqa'
SAMPLE_SIZE = 5000 if TASK == 'docvqa' else 3000
VAL_SIZE = 0.1
SEED = 42

BATCH_SIZE = 2
LEARNING_RATE = 5e-5
NUM_EPOCHS = 2
WEIGHT_DECAY = 0.01

MAX_ANSWER_LENGTH = 64 if TASK == 'docvqa' else 32
MAX_PROMPT_LENGTH = 96
MAX_PATCHES = 1024

HF_TOKEN = ''  # optional
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('TASK =', TASK)
print('SAMPLE_SIZE =', SAMPLE_SIZE)

In [ ]:
def normalize_text(text):
    if text is None:
        return ''
    return ' '.join(str(text).strip().split())


def first_answer(value):
    if isinstance(value, list):
        return value[0] if value else ''
    return value if value is not None else ''


def build_labels(token_ids, pad_token_id):
    return [[-100 if token_id == pad_token_id else token_id for token_id in row] for row in token_ids]


def sample_dataset(dataset, sample_size, seed):
    sample_size = min(sample_size, len(dataset))
    return dataset.shuffle(seed=seed).select(range(sample_size))

## Download and sample source dataset

In [ ]:
if TASK == 'docvqa':
    ds = load_dataset('lmms-lab/DocVQA', 'DocVQA')
    split = ds['train'] if 'train' in ds else ds[next(iter(ds.keys()))]
    split = sample_dataset(split, SAMPLE_SIZE, SEED)
    task_dataset = split.map(
        lambda ex: {
            'task': 'docvqa',
            'image': ex['image'],
            'question': normalize_text(ex['question']),
            'answer': normalize_text(first_answer(ex['answers'])),
        },
        remove_columns=split.column_names,
    )
else:
    ds = load_dataset('HuggingFaceM4/ChartQA')
    split = ds['train'] if 'train' in ds else ds[next(iter(ds.keys()))]
    split = sample_dataset(split, SAMPLE_SIZE, SEED)
    task_dataset = split.map(
        lambda ex: {
            'task': 'chartqa',
            'image': ex['image'],
            'question': normalize_text(ex['query']),
            'answer': normalize_text(first_answer(ex['label'])),
        },
        remove_columns=split.column_names,
    )

print(task_dataset)
task_dataset.select(range(3)).to_pandas()

In [ ]:
dataset = task_dataset.train_test_split(test_size=VAL_SIZE, seed=SEED)
print(dataset)

## Train single-task baseline

In [ ]:
if TASK == 'chartqa':
    MODEL_NAME = 'google/pix2struct-chartqa-base'
    processor = Pix2StructProcessor.from_pretrained(MODEL_NAME)
    model = Pix2StructForConditionalGeneration.from_pretrained(MODEL_NAME)

    def preprocess_batch(batch):
        questions = [normalize_text(question) for question in batch['question']]
        answers = [normalize_text(answer) for answer in batch['answer']]

        model_inputs = processor(
            images=batch['image'],
            text=questions,
            truncation=True,
            max_patches=MAX_PATCHES,
        )

        answer_tokens = processor.tokenizer(
            answers,
            padding='max_length',
            truncation=True,
            max_length=MAX_ANSWER_LENGTH,
        )

        model_inputs['labels'] = build_labels(answer_tokens['input_ids'], processor.tokenizer.pad_token_id)
        return model_inputs

else:
    MODEL_NAME = 'naver-clova-ix/donut-base-finetuned-docvqa'
    processor = DonutProcessor.from_pretrained(MODEL_NAME, use_fast=False)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
    task_start_token = '<s_docvqa>'
    eos_token = processor.tokenizer.eos_token

    def preprocess_batch(batch):
        questions = [normalize_text(question) for question in batch['question']]
        answers = [normalize_text(answer) for answer in batch['answer']]
        prompts = [
            f'{task_start_token}<s_question>{question}</s_question><s_answer>'
            for question in questions
        ]

        rgb_images = [image.convert('RGB') if hasattr(image, 'convert') else image for image in batch['image']]
        image_inputs = processor(
            images=rgb_images,
            return_tensors='pt',
        )
        prompt_tokens = processor.tokenizer(
            prompts,
            add_special_tokens=False,
            padding='max_length',
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
        )

        full_targets = [f'{prompt}{answer}{eos_token}' for prompt, answer in zip(prompts, answers)]
        label_tokens = processor.tokenizer(
            full_targets,
            add_special_tokens=False,
            padding='max_length',
            truncation=True,
            max_length=MAX_PROMPT_LENGTH + MAX_ANSWER_LENGTH,
        )

        labels = build_labels(label_tokens['input_ids'], processor.tokenizer.pad_token_id)
        return {
            'pixel_values': image_inputs['pixel_values'],
            'decoder_input_ids': prompt_tokens['input_ids'],
            'labels': labels,
        }

print('MODEL_NAME =', MODEL_NAME)

In [ ]:
MAP_BATCH_SIZE = 2 if TASK == 'docvqa' else 8

train_dataset = dataset['train'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['train'].column_names,
)

val_dataset = dataset['test'].map(
    preprocess_batch,
    batched=True,
    batch_size=MAP_BATCH_SIZE,
    writer_batch_size=MAP_BATCH_SIZE,
    remove_columns=dataset['test'].column_names,
)

train_dataset.set_format(type='torch')
val_dataset.set_format(type='torch')

print(train_dataset)
print(val_dataset)

In [ ]:
OUTPUT_DIR = WORKDIR / f'{TASK}_baseline_outputs'

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    seed=SEED,
    dataloader_pin_memory=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

## Optional: save outputs to Google Drive

In [ ]:
# Uncomment if you want to save checkpoints to Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "$OUTPUT_DIR" /content/drive/MyDrive/